# Scenario 02: Streaming Responses

**QE Perspective:** The Responses API can return tokens incrementally via streaming. This notebook validates:

- Chunking logic: the stream yields multiple events/chunks.
- Full message capture: iterating the stream and concatenating content yields the complete response.
- Response type and status: streamed response completes successfully and matches non-streamed content semantics.

Configuration: `OGX_BASE_URL` / `MODEL_ID` or `BASE_URL` / `MODEL`. Server assumed running (e.g. `http://localhost:8321`).


## Setup

Load base URL and model from environment; create the OGX client.


In [ ]:
import os
import scripts.helpers  # noqa: F401 — loads .env if present

base_url = os.environ.get("BASE_URL", "http://localhost:8321/v1/")
model = os.environ.get("MODEL")

assert base_url, "OGX_BASE_URL or BASE_URL must be set"
assert model, "MODEL_ID or MODEL must be set"

from ogx_client import OgxClient

client = OgxClient(base_url=base_url)

## Stream iteration and full message capture

Call `responses.create(..., stream=True)` and iterate over the stream. Collect all text deltas to form the full message. Assert we receive multiple chunks and that the concatenated result is non-empty and coherent.


In [ ]:
prompt = "What is the capital of France? Answer in one short sentence."
stream = client.responses.create(
    model=model,
    input=prompt,
    stream=True,
)

chunks = []
full_text = ""
for chunk in stream:
    chunks.append(chunk)
    # Text deltas arrive as OutputTextDelta chunks with a .delta attribute
    delta = getattr(chunk, "delta", None)
    if isinstance(delta, str):
        full_text += delta

assert len(chunks) >= 1, "Expected at least one streamed chunk"
assert full_text.strip(), "Expected non-empty full message from stream"
assert "Paris" in full_text, (
    f"Expected Paris in streamed response, got: {full_text[:200]}"
)

## QE Assertions summary

- At least one chunk received from the stream.
- Concatenated stream content is non-empty.
- Response content is correct (e.g. contains expected answer "Paris").
